Imports

In [ ]:
import sys
import os
import pandas as pd

# Add the parent directory of 'app' to the Python path
sys.path.append(os.path.abspath('D:/GitHub/personal_finances_analysis'))

from app.src.ofx_dataprep import OfxDataPrep

Load data

In [2]:
_path = 'D:/GitHub/DADOS/personal_finances/input'
_output = 'D:/GitHub/DADOS/personal_finances/output'

dataprep = OfxDataPrep()
transactions = dataprep.read_and_prep_data(data_path=_path)

In [3]:
display(transactions.shape, transactions.columns)
transactions.head()

(501, 12)

Index(['type', 'date', 'amount', 'id', 'memo', 'account_id', 'number',
       'institution', 'year', 'year_month', 'in_out', 'is_installment'],
      dtype='object')

,type,date,amount,id,memo,account_id,number,institution,year,year_month,in_out,is_installment
0,Credit Card,2024-12-04,-38.90,674ea97b-5319-4c30-83bf-47b17323cfe5,Cinemark Brasil,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NU PAGAMENTOS S.A.,2024,2024-12,out,False
1,Credit Card,2024-12-02,-14.96,674b9f46-f4c2-4ff9-98f9-722110c83e69,Uber Uber *Trip Help.U,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NU PAGAMENTOS S.A.,2024,2024-12,out,False
2,Credit Card,2024-12-02,-69.58,674c983d-2d23-4a81-9b5f-cd6f4dedbf7d,Vamo Ki Vamo,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NU PAGAMENTOS S.A.,2024,2024-12,out,False
3,Credit Card,2024-12-02,-7.65,674cb742-4e40-4dcb-a7db-98688abb227c,Top Sp Tarif*Descripti,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NU PAGAMENTOS S.A.,2024,2024-12,out,False
4,Credit Card,2024-12-02,-68.05,674c86d2-b258-43ce-b82e-45049ebba24a,Horti Fruti Ortencia,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NU PAGAMENTOS S.A.,2024,2024-12,out,False


EDA

In [4]:
transactions.dtypes

type                 object
date                 object
amount              float64
id                   object
memo                 object
account_id           object
number               object
institution          object
year                  int32
year_month        period[M]
in_out               object
is_installment         bool
dtype: object

In [5]:
transactions.isnull().sum()

type              0
date              0
amount            0
id                0
memo              0
account_id        0
number            0
institution       0
year              0
year_month        0
in_out            0
is_installment    0
dtype: int64

In [6]:
transactions.head(4)

,type,date,amount,id,memo,account_id,number,institution,year,year_month,in_out,is_installment
0,Credit Card,2024-12-04,-38.90,674ea97b-5319-4c30-83bf-47b17323cfe5,Cinemark Brasil,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NU PAGAMENTOS S.A.,2024,2024-12,out,False
1,Credit Card,2024-12-02,-14.96,674b9f46-f4c2-4ff9-98f9-722110c83e69,Uber Uber *Trip Help.U,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NU PAGAMENTOS S.A.,2024,2024-12,out,False
2,Credit Card,2024-12-02,-69.58,674c983d-2d23-4a81-9b5f-cd6f4dedbf7d,Vamo Ki Vamo,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NU PAGAMENTOS S.A.,2024,2024-12,out,False
3,Credit Card,2024-12-02,-7.65,674cb742-4e40-4dcb-a7db-98688abb227c,Top Sp Tarif*Descripti,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NU PAGAMENTOS S.A.,2024,2024-12,out,False


In [ ]:
display(transactions.query('type == "Credit Card"')['in_out'].value_counts())
display(transactions.query('type == "Credit Card" and in_out == "in"').groupby(['year_month'])['amount'].sum())

cc_total_out = transactions.query('type == "Credit Card" and in_out == "out"')\
                            .groupby(['year_month'])['amount']\
                            .sum()\
                            .reset_index(name='cc_total_out')

cc_total_out_no_installments = transactions.query('type == "Credit Card" and in_out == "out" and is_installment == False')\
                                            .groupby(['year_month'])['amount']\
                                            .sum()\
                                            .reset_index(name='cc_total_out_no_installments')

cc_out_installment = transactions.query('type == "Credit Card" and in_out == "out" and is_installment == True')\
                                    .groupby(['year_month'])['amount']\
                                    .sum()\
                                    .reset_index(name='cc_out_installment')

result = cc_total_out.merge(cc_total_out_no_installments, on="year_month", how="left")
result = result.merge(cc_out_installment, on="year_month", how="left")
result.fillna(0, inplace=True)
display(result)

#transactions.query('type == "Credit Card" and in_out == "out"').sort_values(by='amount', ascending=False)

in_out
out    370
Name: count, dtype: int64

Series([], Freq: M, Name: amount, dtype: float64)

,year_month,cc_total_out,cc_total_out_no_installments,cc_out_installment
0,2024-11,-3196.73,-2472.16,-724.57
1,2024-12,-3956.23,-2520.14,-1436.09
2,2025-01,-4182.86,-2349.06,-1833.80
3,2025-02,-5942.75,-4139.51,-1803.24
4,2025-03,-5064.22,-3511.79,-1552.43
5,2025-04,-193.14,-193.14,0.00


In [ ]:
transactions.query('type == "Bank"').shape